## Churn Data — Feature Engineering

### By:
jdg

### Date:
2026-03-03

### Description:

Builds a reusable scikit-learn preprocessing pipeline that transforms the
primary data (`churn_primary.parquet`) into model-ready features.

Key decisions from the variable analysis (Step 3):
- **Drop**: `gender`, `PhoneService`, `TotalCharges`
- **Consolidate**: "No internet/phone service" → "No"
- **Add**: `is_new_customer` binary (tenure ≤ 6 months)
- **Preserve** class imbalance (~26% churn) — no SMOTE

The pipeline is built **unfitted** here. In Step 5, it will be wrapped
with a model into a full `Pipeline` and used inside cross-validation.

## 📚 Import libraries

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Add src to path so we can import project modules
sys.path.insert(0, str(Path("../../src").resolve()))

from data.transformation import build_feature_pipeline, load_feature_config

### Load configuration

In [ ]:
config = load_feature_config(Path("../../conf/data_preparation/features.yml"))
config

## 💾 Load data

In [ ]:
PRIMARY_PATH = Path("../../") / config["input_path"]

df = pd.read_parquet(PRIMARY_PATH, dtype_backend="numpy_nullable")
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

## 👷 Separate X and y

In [ ]:
target = config["target"]

X = df.drop(columns=[target])
y = df[target].astype(int)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print("\nClass distribution:")
print(y.value_counts())
print(f"\nChurn rate: {y.mean():.1%}")

## 🔧 Build preprocessing pipeline

In [ ]:
pipeline = build_feature_pipeline(config)
pipeline

## ⚙️ Fit & transform (full data, for inspection only)

In [ ]:
X_transformed = pipeline.fit_transform(X)
print(f"Output shape: {X_transformed.shape}")
X_transformed.head()

### Verify output

In [ ]:
print(f"Shape: {X_transformed.shape}")
print(f"NaN count: {X_transformed.isna().sum().sum()}")
print(f"\nDtypes:\n{X_transformed.dtypes.value_counts()}")
print(f"\nColumns:\n{list(X_transformed.columns)}")

In [ ]:
X_transformed.describe()

### Inspect sub-pipelines

In [ ]:
# Numeric imputer statistics
numeric_imputer = pipeline.named_transformers_["numeric"]["imputer"]
print("Numeric imputer statistics (median):")
for col, stat in zip(config["numeric_columns"], numeric_imputer.statistics_, strict=False):
    print(f"  {col}: {stat}")

# OHE categories for consolidate_ohe
consolidate_encoder = pipeline.named_transformers_["consolidate_ohe"]["encoder"]
print(f"\nConsolidate OHE categories: {consolidate_encoder.categories_}")

# OHE categories for multi_ohe
multi_encoder = pipeline.named_transformers_["multi_ohe"]["encoder"]
print(f"\nMulti OHE categories: {multi_encoder.categories_}")

# Ordinal encoding for contract
contract_encoder = pipeline.named_transformers_["contract"]["encoder"]
print(f"\nContract ordinal categories: {contract_encoder.categories_}")

### Verify is_new_customer

In [ ]:
# Cross-tab is_new_customer with tenure ranges
tenure_bins = pd.cut(
    X["tenure"].astype(float),
    bins=[0, 6, 12, 24, 48, 72],
    labels=["0-6", "7-12", "13-24", "25-48", "49-72"],
    include_lowest=True,
)
cross = pd.crosstab(tenure_bins, X_transformed["is_new_customer"])
print("is_new_customer cross-tab with tenure ranges:")
cross

## 💾 Save artifacts

In [ ]:
output_path = Path("../../") / config["feature_output_path"]
output_path.mkdir(parents=True, exist_ok=True)

features_path = output_path / "features.parquet"
target_path = output_path / "target.parquet"

X_transformed.to_parquet(features_path, index=False)
y.to_frame().to_parquet(target_path, index=False)

print(f"Saved features: {features_path} ({X_transformed.shape})")
print(f"Saved target:   {target_path} ({y.shape[0]:,} rows)")

## 📊 Analysis of Results and Conclusions

- **Input**: 7,194 rows × 19 features (after dropping target)
- **Output**: 7,194 rows × 22 features, all float64, zero NaN
- **Dropped**: `gender` (low predictive power), `PhoneService` (low predictive power),
  `TotalCharges` (multicollinear with tenure, r=0.82)
- **Consolidation**: "No internet/phone service" → "No" reduced 7 columns from 3 levels
  to 2 (binary after OHE with `drop='if_binary'`)
- **New feature**: `is_new_customer` captures the high-churn short-tenure segment
- **Imputation**: median for numeric, mode for boolean/categorical — all inside the
  pipeline to prevent data leakage during cross-validation
- The pipeline is built **unfitted** and importable from `src/data/transformation`
  for direct use in Step 5's model pipeline

## 💡 Proposals and Ideas

- Proceed to `5-models/`: wrap `build_feature_pipeline()` + model in a full
  `Pipeline`, evaluate with `cross_val_score` using `StratifiedKFold`
- Models to try: Logistic Regression (baseline), Random Forest, Gradient Boosting
- Use `class_weight='balanced'` to handle the ~26% churn imbalance
- Hyperparameter tuning via `GridSearchCV` or `RandomizedSearchCV`